In [3]:
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path="../.env")

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# === 컴포넌트 생성 ===
# 프롬프트 템플릿: 시스템 메시지와 사용자 메시지를 구조화합니다.
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 {topic} 전문가입니다. 간결하게 답변하세요."),
    ("human", "{question}")
])

# 모델: Gemini 2.5 Flash를 사용합니다.
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    google_api_key=GOOGLE_API_KEY
)

# 출력 파서: 모델 응답에서 텍스트만 추출합니다.
parser = StrOutputParser()

# === LCEL 파이프 연산자로 체인 구성 ===
chain = prompt | model | parser

In [5]:
# === invoke: 단일 입력 실행 ===
result = chain.invoke({
    "topic": "번역",
    "question": "다음 문장을 영어로 번역하세요: 어댑터즈는 스타트업코드에서 제공하는 개발 책 서빙 서비스입니다."
})
print(result)

Adapters is a developer book serving service provided by Startupcode.


In [6]:
# === 같은 체인, 다른 입력 ===
# 요약 작업
result2 = chain.invoke({
    "topic": "요약",
    "question": "다음 문장을 한 줄로 요약하세요: Startupcode is a company that specializes in developer education. It operates a practice-oriented curriculum and provides textbooks to students through Adapterz."
})
print(f"요약: {result2}")

# 일반 질문
result3 = chain.invoke({
    "topic": "Python",
    "question": "리스트와 튜플의 차이를 설명하세요."
})
print(f"답변: {result3}")

요약: Startupcode는 Adapterz를 통해 교재를 제공하고 실습 중심의 교육 과정을 운영하는 개발자 교육 전문 회사입니다.
답변: 리스트는 변경 가능(mutable)하고 튜플은 변경 불가능(immutable)합니다.


In [7]:
# === 하드코딩 방식 (LangChain 없이) ===
from google import genai

client = genai.Client(api_key=GOOGLE_API_KEY)

topic = "번역"
question = "다음 문장을 영어로 번역하세요: 어댑터즈는 스타트업코드에서 제공하는 개발 책 서빙 서비스입니다."

# 프롬프트를 f-string으로 직접 조합
prompt_text = f"당신은 {topic} 전문가입니다. 간결하게 답변하세요.\n\n{question}"

# API 직접 호출
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt_text
)

# 응답에서 텍스트 수동 추출
result = response.text
print(result)

Adapters is a development book delivery service provided by Startup Code.
